In [ ]:
#Basic setup for a laser pulse interation with a solid-density plasma layer 
#for results see fig. 6 in arXiv:2302.01893
import sys
import pipic
from pipic.extensions import moving_window
from pipic import consts,types
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from numba import cfunc, carray, types as nbt
import numpy as np
from matplotlib import animation
from IPython.display import HTML, display, clear_output, Image

In [ ]:
#===========================SIMULATION PARAMETERS===========================
n0 = 8e18 # Electron number density [1/cm^3]
omega_p = np.sqrt(np.pi*4*consts.electron_charge**2*n0/consts.electron_mass) # plasma frequency [1/s]
wl = 1e-4 # laser wavelength [cm] 
wl_p = 2*np.pi*consts.light_velocity/omega_p # plasma wavelength [cm]

In [ ]:
#===========================SIMULATION INITIALIZATION===========================
nx, xmin, xmax = 2**13,  -100*wl, 100*wl 
dx = (xmax - xmin)/nx # 10 timesteps per laser cycle
timestep = dx/consts.light_velocity/4
sim=pipic.init(solver='fourier_boris',nx=nx,xmin=xmin,xmax=xmax)


In [ ]:
#=================================FIELD========================================
# laser (radial) frequency
omega = 2 * np.pi * consts.light_velocity / wl # [1/s]
a0 = 10 # [unitless] (this a high a0)
pulseWidth_x = 10*wl # [cm] (longitudinal width of laser)
E0 = a0 * consts.electron_mass * consts.light_velocity * omega / consts.electron_charge # Field amplitude [statV/cm] 
pos = 0 # initial position of the pulse center [cm]
@cfunc(types.field_loop_callback)
def initiate_field_callback(ind, r, E, B, data_double, data_int):
    if data_int[0] == 0:      
        x = r[0] - pos
        k = 2*np.pi/wl
        gp = np.real(E0*np.exp(-1j*(k*x))*np.exp(-x**2/(2*pulseWidth_x**2)))
        # z-polarized 
        E[2] = gp       
        B[1] = -gp     

#-----------------------initiate field and plasma-------------------------
data_int = np.zeros((1, ), dtype=np.intc) # data for passing the iteration number
#sim.field_loop(handler=initiate_field_callback.address, data_int=pipic.addressof(data_int), use_omp=True)

#sim.fourier_solver_settings(divergence_cleaning=1)


sim.fourier_solver_settings(divergence_cleaning=1, sin2_kfilter=-1)
sim.advance(time_step=0, number_of_iterations=1,use_omp=True)
sim.field_loop(handler=initiate_field_callback.address, data_int=pipic.addressof(data_int),
                use_omp=True)
sim.advance(time_step=0, number_of_iterations=1,use_omp=True)
sim.fourier_solver_settings(divergence_cleaning=0, sin2_kfilter=-1)


In [ ]:
#=================================PLASMA PROFILE========================================
# The plasma profile consist of an upramp with length xmax followed by a constant section with length 2*xmax
debye_length = 1e-2 # [cm] 
temperature = 1e-6 * 4 * np.pi * n0 * consts.electron_charge ** 2 * debye_length ** 2 # [erg/kB] 
particles_per_cell = 5
start_upramp = xmax
end_upramp = start_upramp + xmax #10*wl
end_of_plasma = end_upramp + xmax*4
@cfunc(types.add_particles_callback)
def density_profile(r, data_double, data_int):
    if r[0] < start_upramp:
        return 0
    elif r[0]>start_upramp and r[0] < end_upramp: 
        return n0*((r[0]-start_upramp)/(end_upramp-start_upramp))
    elif r[0] >= end_upramp and r[0] < end_of_plasma:
        return n0
    else:
        return 0 
    

# This line is just for initiating the electron species, so that the algorithm knows that there is a species called electron. 
# The actual addition of particles will be done by the extension. The density_profile function will be passed to the extension.
sim.add_particles(name='electron', number=1,#particles_per_cell),
                charge=consts.electron_charge, mass=consts.electron_mass,
                temperature=temperature, density=density_profile.address,)

In [ ]:
#=================================ADDING THE EXTENSION========================================
thickness = 2**3 # thickness (in dx) of the area where the density and field is restored/removed in the moving window
window_speed = consts.light_velocity #speed of moving window
# defining the particle handler
density_handler_adress = moving_window.handler(sim.simulation_box(),
                                               thickness=thickness,
                                               particles_per_cell=particles_per_cell,
                                               temperature=temperature,
                                               density_profile=density_profile.address,
                                               velocity=window_speed,
                                               axis = 'x')
# defining the field handler
field_handler_adress = moving_window.field_handler(sim.simulation_box(),thickness=thickness,velocity=window_speed,axis='x')

# adding the handlers
sim.add_handler(name=moving_window.name, 
                subject='electron,cells',
                handler=density_handler_adress,
                field_handler=field_handler_adress,
                data_int=pipic.addressof(data_int),)

In [ ]:

#=================================OUTPUT=======================================
#-------------------------preparing output of fields and density-----------------------------
Ex = np.zeros((nx,), dtype=np.double) 
rho = np.zeros((nx,), dtype=np.double) 

#------------------get functions-----------------------------------------------
# Function for looping through particles, calculating density and saving it in 'rho'
@cfunc(types.particle_loop_callback)
def get_density(r, p, w, id, data_double, data_int):  
        ix = int(rho.shape[0]*(r[0] - xmin)/(xmax - xmin))
        data = carray(data_double, rho.shape, dtype=np.double)
        if ix < rho.shape[0]:
            data[ix] += w[0]/dx

# Function for reading Ex field and saving it in 'Ex' 
@cfunc(types.field_loop_callback)
def get_field_Ex(ind, r, E, B, data_double, data_int):       
    _E = carray(data_double, Ex.shape, dtype=np.double)
    _E[ind[0],] = E[2]


fig,ax = plt.subplots(2,1)

# Ex-field plot
ax[1].set_xlim([xmin, xmax])
ax[1].set_ylim([-E0, E0])
ax[1].set(xlabel='$x$ (cm)', ylabel='$E_x$ (cgs units)')
x_axis = np.linspace(xmin, xmax, Ex.shape[0])
plot_Ex, = ax[1].plot(x_axis, Ex)

# Density plot
ax[0].set_title('$\partial N / \partial x \partial p_x$ (s g$^{-1}$cm$^{-2}$)')
ax[0].set(ylabel='$p_x$ (cm g/s)')
ax[0].set_ylim(0,n0*10)
plot_rho, = ax[0].plot(x_axis,rho)

In [ ]:
# This cell may take a while (depending on your processing power). If it runs to slow, reduce the number of frames or number_of_iterations.
# Alternatively: go and have a coffee :)

frames = 200
counter = 0
def animate(i):


    sim.advance(time_step=timestep, number_of_iterations=300,use_omp=True)

    rho.fill(0)
    sim.particle_loop(name='electron', handler=get_density.address, data_double=pipic.addressof(rho))
    plot_rho.set_ydata(rho)

    sim.field_loop(handler=get_field_Ex.address, data_double=pipic.addressof(Ex),use_omp = True)
    plot_Ex.set_ydata(Ex)

    
    global counter
    clear_output()
    if counter <= frames:
        display(HTML('<pre> Progress: ' + "{:.2f}".format(100*counter/frames) + '%</pre>'), display_id = True)
    counter += 1
    return
    

ani = animation.FuncAnimation(fig, animate,frames=frames, interval = 40)
with open('pipic_performance.txt', 'r') as f:
    print(f.read())

html = HTML(ani.to_jshtml())
display(html)
plt.close()

# re-run this cell to propagate the simulation further